In [1]:
# ==============================================================================
# UNAD - PROGRAMACIÓN
# FASE 4: COMPONENTE PRÁCTICO - PRÁCTICAS SIMULADAS
# DESARROLLO INDIVIDUAL - MANEJO DE EXCEPCIONES Y ARCHIVOS EN POO
# ==============================================================================

import os

# 1. DEFINICIÓN DE EXCEPCIONES PERSONALIZADAS
class SaldoInsuficienteError(Exception):
    def __init__(self, saldo_actual, monto_solicitado):
        self.message = f"Error: Saldo insuficiente. Saldo disponible: ${saldo_actual}, Monto solicitado: ${monto_solicitado}"
        super().__init__(self.message)

class MontoInvalidoError(Exception):
    def __init__(self, monto):
        self.message = f"Error: El monto ingresado (${monto}) debe ser mayor a cero."
        super().__init__(self.message)

class AutenticacionError(Exception):
    def __init__(self):
        self.message = "Error: Autenticación fallida. El número de cuenta o PIN es incorrecto."
        super().__init__(self.message)

# 2. SISTEMA DE FLUJO DE DATOS (Guardado de errores en archivo de texto)
def registrar_error_en_archivo(mensaje_error):
    try:
        with open("log_errores.txt", "a", encoding="utf-8") as archivo:
            archivo.write(f"[REGISTRO] {mensaje_error}\n")
    except IOError:
        print("No se pudo escribir en el archivo de registro de errores.")

# 3. CLASE PRINCIPAL: CUENTA BANCARIA
class CuentaBancaria:
    def __init__(self, numero_cuenta, pin, titular, saldo_inicial=0.0):
        self.__numero_cuenta = numero_cuenta  
        self.__pin = pin                      
        self.titular = titular                
        self.__saldo = float(saldo_inicial)   

    def verificar_credenciales(self, numero_cuenta, pin):
        return self.__numero_cuenta == numero_cuenta and self.__pin == pin

    def consultar_saldo(self):
        return self.__saldo

    def depositar(self, monto):
        if monto <= 0:
            raise MontoInvalidoError(monto)
        self.__saldo += monto
        print(f"✔️ Depósito exitoso. Nuevo saldo: ${self.__saldo}")

    def retirar(self, monto):
        if monto <= 0:
            raise MontoInvalidoError(monto)
        if monto > self.__saldo:
            raise SaldoInsuficienteError(self.__saldo, monto)
        self.__saldo -= monto
        print(f"✔️ Retiro exitoso. Saldo restante: ${self.__saldo}")

# 4. INTERFAZ DE USUARIO Y CONTROL DE FLUJO
def ejecutar_cajero():
    cuenta_usuario = CuentaBancaria(numero_cuenta="123456", pin="9876", titular="Félix Dávila", saldo_inicial=500000.0)
    print("=== BIENVENIDO AL SISTEMA DE CAJERO AUTOMÁTICO (UNAD FASE 4) ===")
    intentos = 3
    autenticado = False
    
    while intentos > 0 and not autenticado:
        try:
            print(f"\n[Intentos restantes: {intentos}]")
            u_cuenta = input("Ingrese su número de cuenta: ").strip()
            u_pin = input("Ingrese su PIN de seguridad: ").strip()
            if not cuenta_usuario.verificar_credenciales(u_cuenta, u_pin):
                raise AutenticacionError()
            autenticado = True
            print(f"\n🎉 ¡Bienvenido(a) {cuenta_usuario.titular}!")
        except AutenticacionError as e:
            print(e)
            registrar_error_en_archivo(e.message)
            intentos -= 1
            if intentos == 0:
                print("❌ Cuenta bloqueada por seguridad tras múltiples intentos fallidos.")
                registrar_error_en_archivo("Cuenta bloqueada por exceso de intentos.")
                return

    while autenticado:
        print("\n--- MENÚ DE OPCIONES ---")
        print("1. Consultar Saldo")
        print("2. Realizar Depósito")
        print("3. Realizar Retiro")
        print("4. Salir")
        try:
            opcion = input("Seleccione una opción (1-4): ").strip()
            if opcion == "1":
                print(f"💰 Su saldo actual es: ${cuenta_usuario.consultar_saldo()}")
            elif opcion == "2":
                try:
                    monto = float(input("Ingrese el monto a depositar: $"))
                    cuenta_usuario.depositar(monto)
                except ValueError:
                    print("Error: Debe ingresar un valor numérico.")
                except MontoInvalidoError as e:
                    print(e)
            elif opcion == "3":
                try:
                    monto = float(input("Ingrese el monto a retirar: $"))
                    cuenta_usuario.retirar(monto)
                except ValueError:
                    print("Error: Debe ingresar un valor numérico.")
                except (MontoInvalidoError, SaldoInsuficienteError) as e:
                    print(e)
            elif opcion == "4":
                print("👋 Gracias por utilizar nuestros servicios virtuales. ¡Hasta pronto!")
                break
            else:
                print("⚠ Opción no válida.")
        except Exception as e:
            print(f"Error imprevisto: {e}")

if __name__ == "__main__":
    ejecutar_cajero()

=== BIENVENIDO AL SISTEMA DE CAJERO AUTOMÁTICO (UNAD FASE 4) ===

[Intentos restantes: 3]
Error: Autenticación fallida. El número de cuenta o PIN es incorrecto.
No se pudo escribir en el archivo de registro de errores.

[Intentos restantes: 2]
Error: Autenticación fallida. El número de cuenta o PIN es incorrecto.

[Intentos restantes: 1]

🎉 ¡Bienvenido(a) Félix Dávila!

--- MENÚ DE OPCIONES ---
1. Consultar Saldo
2. Realizar Depósito
3. Realizar Retiro
4. Salir
💰 Su saldo actual es: $500000.0

--- MENÚ DE OPCIONES ---
1. Consultar Saldo
2. Realizar Depósito
3. Realizar Retiro
4. Salir
👋 Gracias por utilizar nuestros servicios virtuales. ¡Hasta pronto!
